# Module 1: Exercise solutions

Worked answers to the five exercises at the end of `notebook.ipynb`. Try them yourself first.

The first cell re-runs the Module 1 notebook so that `token_f1`, `corpus_bleu` and the other functions are available here.

In [1]:
%run ./notebook.ipynb

## Exercise 1: token F1 by hand

Prediction "barack obama" against reference "president barack obama".

- Shared tokens: barack, obama, so common = 2
- Precision = 2 / 2 = 1.0, because everything predicted is in the reference
- Recall = 2 / 3 = 0.667, because "president" is missing
- F1 = 2 x 1.0 x 0.667 / (1.0 + 0.667) = 0.8

The answer loses points only for leaving something out, not for adding anything wrong.

In [2]:
f1 = token_f1("barack obama", ["president barack obama"])
print(f"token F1 = {f1:.3f}")
assert abs(f1 - 0.8) < 1e-9

token F1 = 0.800


## Exercise 2: the brevity penalty

Both words of "the cat" appear in the reference, so unigram and bigram precision are both perfect. The output is far shorter than the reference, though: c = 2 tokens against r = 6.

BP = exp(1 - r / c) = exp(1 - 6 / 2) = exp(-2) = 0.135

The penalty cuts BLEU from 100 to about 13.5. Without it, a model could score perfectly by answering with one safe word.

In [3]:
res = corpus_bleu(["the cat"], ["the cat sat on the mat"], max_n=2)
print(f"precisions = {[round(p, 1) for p in res['precisions']]}")
print(f"brevity penalty = {res['bp']:.4f}")
print(f"BLEU = {res['bleu']:.2f}")
assert abs(res["bp"] - math.exp(-2)) < 1e-9

precisions = [100.0, 100.0]
brevity penalty = 0.1353
BLEU = 13.53


## Exercise 3: clipping

Prediction "the the the the" against reference "the cat".

- **Without clipping:** all 4 predicted tokens are "the", which appears in the reference, so p1 = 4 / 4 = 1.0.
- **With clipping:** "the" appears only once in the reference, so it earns credit once. p1 = 1 / 4 = 0.25.

Clipping is what stops repetition from gaming the score.

In [4]:
pred, ref = "the the the the".split(), "the cat".split()
unclipped = sum(1 for tok in pred if tok in ref) / len(pred)
clipped = sum((ngrams(pred, 1) & ngrams(ref, 1)).values()) / len(pred)
print(f"p1 without clipping = {unclipped:.2f}")
print(f"p1 with clipping    = {clipped:.2f}")

p1 without clipping = 1.00
p1 with clipping    = 0.25


## Exercise 4: your own RAG answers

There is no single answer here. The template below scores five question-answer pairs from any RAG system. Replace the examples with your own and compare the scores with your own judgement.

Things to look for:

- Correct answers that score low because they are worded differently from the reference
- Wrong answers that score high because they repeat the question's words
- Long, correct answers that lose precision because of extra explanation

In [5]:
my_rag_answers = [
    # (question, RAG system answer, list of acceptable references)
    ("Which retriever combines dense and BM25 scores?", "Hybrid retrieval with reciprocal rank fusion", ["hybrid retrieval", "reciprocal rank fusion"]),
    ("What does a cross-encoder do?", "It re-scores each query and passage pair together", ["reranks passages by scoring query and passage jointly"]),
    ("What triggers the web-search fallback?", "Low relevance scores for the retrieved documents", ["when retrieved documents are graded irrelevant"]),
    ("What metric rewards ranking the first relevant result high?", "MRR", ["mean reciprocal rank", "MRR"]),
    ("What does BM25 match on?", "Exact keywords", ["keyword overlap", "exact terms"]),
]
print(f"{'question':58} {'EM':>3} {'F1':>6}")
for q, ans, refs in my_rag_answers:
    print(f"{q[:56]:58} {exact_match(ans, refs):>3} {token_f1(ans, refs):>6.3f}")

question                                                    EM     F1
Which retriever combines dense and BM25 scores?              0  0.667
What does a cross-encoder do?                                0  0.375
What triggers the web-search fallback?                       0  0.333
What metric rewards ranking the first relevant result hi     1  1.000
What does BM25 match on?                                     0  0.500


## Exercise 5: what a human grader needs

Case 1 in Part 5 scored 0 on both F1 and ROUGE-L, yet the prediction is correct. A human grader sees that:

- "medication" and "drug" mean the same thing
- "twice daily" and "two times a day" mean the same thing
- "with food" and "alongside meals" mean the same thing

So the grader needs knowledge of **meaning**, not just shared words. That is synonymy, paraphrase and some domain knowledge. Word-overlap metrics have none of this.

There are two common ways to add it:

1. **Embedding similarity:** compare the meaning of the two answers as vectors. This is cheap but coarse, and it can still be fooled by negation.
2. **An LLM as the judge:** ask a model to grade with a rubric. This is more flexible, but it needs its own checks for bias and agreement with human grades.

Module 3 builds and tests the second option.